# ESM-2 Protein Embeddings with BioNeMo NIM

This notebook demonstrates how to serve ESM-2 protein language model using BioNeMo NIM and Ray Serve.

**ESM-2** (Evolutionary Scale Modeling) is a state-of-the-art protein language model that provides:
- Protein sequence embeddings (1280-dim for ESM2-650M)
- Per-residue representations
- Contact map predictions for structure

## Prerequisites

1. NGC API key (get one at https://ngc.nvidia.com)
2. GPU with at least 40GB VRAM (A100-40GB or H100)
3. Docker with NVIDIA runtime installed

## 1. Setup

In [ ]:
import os
import sys

# Add parent directory to path for imports
sys.path.insert(0, ".")
sys.path.insert(0, "../../ray/genai")  # For cluster utilities

# Check NGC API key
if not os.environ.get("NGC_API_KEY"):
    print("WARNING: NGC_API_KEY not set. Set it before starting the NIM container.")
    print("export NGC_API_KEY='your-api-key'")

In [ ]:
# Configuration
ESM2_NIM_HOST = "localhost"
ESM2_NIM_PORT = 8001

# Example protein sequences
INSULIN_A_CHAIN = "GIVEQCCTSICSLYQLENYCN"
INSULIN_B_CHAIN = "FVNQHLCGSHLVEALYLVCGERGFFYTPKT"
HEMOGLOBIN_ALPHA = "MVLSPADKTNVKAAWGKVGAHAGEYGAEALERMFLSFPTTKTYFPHFDLSH"

## 2. Start ESM-2 NIM Container

Run this in a terminal (or uncomment the cell below):

```bash
export NGC_API_KEY='your-api-key'
./scripts/start_nims.sh esm2
```

In [ ]:
# Uncomment to start ESM-2 NIM from notebook
# !./scripts/start_nims.sh esm2

In [ ]:
# Check if NIM is ready
import httpx

def check_esm2_health():
    try:
        response = httpx.get(f"http://{ESM2_NIM_HOST}:{ESM2_NIM_PORT}/v1/health/ready", timeout=10)
        if response.status_code == 200:
            print("✅ ESM-2 NIM is ready!")
            return True
    except Exception as e:
        print(f"❌ ESM-2 NIM not ready: {e}")
    return False

check_esm2_health()

## 3. Direct NIM API Usage

First, let's use the NIM directly without Ray Serve.

In [ ]:
from utils.nim_client import NIMClient, DEFAULT_ENDPOINTS
import asyncio

async def get_protein_embeddings(sequences):
    """Get embeddings for protein sequences."""
    endpoint = DEFAULT_ENDPOINTS["esm2"]
    
    async with NIMClient(endpoint) as client:
        result = await client.predict({
            "sequences": sequences,
            "include_embeddings": True,
        })
    return result

# Get embeddings for insulin chains
result = await get_protein_embeddings([INSULIN_A_CHAIN, INSULIN_B_CHAIN])

print(f"Number of sequences: {len(result.get('results', []))}")
if 'results' in result:
    for i, r in enumerate(result['results']):
        emb_shape = len(r.get('embedding', [])) if 'embedding' in r else 'N/A'
        print(f"  Sequence {i+1}: embedding dimension = {emb_shape}")

## 4. Initialize Ray Cluster

In [ ]:
# Import cluster utilities (same as genai demos)
try:
    from utils.cluster import init_ray, ClusterMode, shutdown_ray
except ImportError:
    # Fallback if genai utils not available
    import ray
    from enum import Enum
    
    class ClusterMode(Enum):
        LOCAL = "local"
        ANYSCALE = "anyscale"
    
    def init_ray(mode=ClusterMode.LOCAL):
        if not ray.is_initialized():
            if mode == ClusterMode.ANYSCALE:
                ray.init(address="auto")
            else:
                ray.init()
        print(f"Ray Dashboard: {ray.dashboard_url}")
    
    def shutdown_ray():
        ray.shutdown()

In [ ]:
# Initialize Ray in local mode
CLUSTER_MODE = ClusterMode.LOCAL  # Change to ANYSCALE for production

init_ray(mode=CLUSTER_MODE)

## 5. Deploy ESM-2 with Ray Serve

In [ ]:
from ray import serve
from deployments.esm2_deployment import ESM2Deployment

# Create and deploy ESM-2
esm2_deployment = ESM2Deployment.bind(
    nim_host=ESM2_NIM_HOST,
    nim_port=ESM2_NIM_PORT,
)

handle = serve.run(esm2_deployment, name="esm2", route_prefix="/esm2")

print("\nESM-2 deployed!")
print(f"  Endpoint: http://localhost:8000/esm2")
print(f"  Dashboard: http://localhost:8265")

## 6. Test the API

In [ ]:
import requests
import json

# Test embeddings endpoint
response = requests.post(
    "http://localhost:8000/esm2",
    json={
        "action": "embeddings",
        "sequences": [INSULIN_A_CHAIN, INSULIN_B_CHAIN],
    }
)

print("ESM-2 Embeddings Response:")
result = response.json()

# Print summary (embeddings are large)
if "results" in result:
    for i, r in enumerate(result["results"]):
        print(f"  Sequence {i+1}: {len(r.get('embedding', []))} dimensions")
elif "error" in result:
    print(f"  Error: {result['error']}")

print(f"\nLatency: {result.get('_metadata', {}).get('latency_ms', 'N/A'):.1f}ms")

In [ ]:
# Test contact/structure prediction
response = requests.post(
    "http://localhost:8000/esm2",
    json={
        "action": "contacts",
        "sequence": HEMOGLOBIN_ALPHA,
    }
)

print("ESM-2 Contact Prediction:")
result = response.json()

if "results" in result and result["results"]:
    contacts = result["results"][0].get("contacts")
    if contacts:
        print(f"  Contact map shape: {len(contacts)} x {len(contacts[0]) if contacts else 0}")
elif "error" in result:
    print(f"  Error: {result['error']}")

## 7. Batch Inference with Ray Data

In [ ]:
import ray
from ray import data as ray_data
import numpy as np
from utils.nim_client import SyncNIMClient, DEFAULT_ENDPOINTS

class ESM2BatchPredictor:
    """Batch predictor for ESM-2 embeddings with Ray Data."""
    
    def __init__(self, nim_host="localhost", nim_port=8001):
        endpoint = DEFAULT_ENDPOINTS["esm2"]
        endpoint.host = nim_host
        endpoint.port = nim_port
        self.client = SyncNIMClient(endpoint)
    
    def __call__(self, batch):
        sequences = batch["sequence"].tolist()
        
        result = self.client.predict({
            "sequences": sequences,
            "include_embeddings": True,
        })
        
        # Extract embeddings
        embeddings = []
        for r in result.get("results", []):
            emb = r.get("embedding", [])
            embeddings.append(emb)
        
        return {
            "sequence": batch["sequence"],
            "embedding": np.array(embeddings, dtype=object),
        }

In [ ]:
# Create a sample dataset of protein sequences
protein_data = [
    {"sequence": INSULIN_A_CHAIN, "name": "Insulin A"},
    {"sequence": INSULIN_B_CHAIN, "name": "Insulin B"},
    {"sequence": HEMOGLOBIN_ALPHA[:50], "name": "Hemoglobin (truncated)"},
    {"sequence": "MKTVRQERLKSIVRILERSKEPVSGAQL", "name": "Sample 1"},
    {"sequence": "AAEELSVSRQVIVQDIAYLRSLGYNIVATPRGYVLAGG", "name": "Sample 2"},
]

# Create Ray Dataset
ds = ray_data.from_items(protein_data)
print(f"Dataset size: {ds.count()} sequences")

In [ ]:
# Run batch inference
results = ds.map_batches(
    ESM2BatchPredictor,
    fn_constructor_kwargs={
        "nim_host": ESM2_NIM_HOST,
        "nim_port": ESM2_NIM_PORT,
    },
    batch_size=2,  # Small batch for demo
    num_cpus=1,
    concurrency=1,
)

# Collect results
output = results.take_all()
print(f"\nProcessed {len(output)} sequences")
for row in output:
    emb_len = len(row['embedding']) if row['embedding'] is not None else 0
    print(f"  {row['sequence'][:20]}... -> {emb_len} dim embedding")

## 8. Cleanup

In [ ]:
# Shutdown Ray Serve
serve.shutdown()
print("Ray Serve shutdown complete")

In [ ]:
# Shutdown Ray
shutdown_ray()
print("Ray cluster shutdown complete")

In [ ]:
# Optionally stop the NIM container
# !./scripts/stop_nims.sh esm2

## Next Steps

- **Multi-model gateway**: See `multi_model_gateway.ipynb` for deploying all BioNeMo models
- **Evo2 for genomics**: See `evo2_serving.ipynb` for DNA sequence generation
- **Geneformer**: See `geneformer_serving.ipynb` for single-cell analysis
- **Production deployment**: Configure Anyscale for autoscaling

## Resources

- [ESM-2 Paper](https://www.science.org/doi/10.1126/science.ade2574)
- [BioNeMo Documentation](https://docs.nvidia.com/bionemo-framework/latest/)
- [Ray Serve Documentation](https://docs.ray.io/en/latest/serve/index.html)